# 16_E8 - Al-Kafri official pairing and mask decode rescue

Objetivo: intentar rescatar el dataset axial Al-Kafri/Sudirman usando metadata y estructura oficial, antes de descartarlo para segmentación axial.

Este notebook **no entrena modelos**. Solo:
- busca documentos, CSVs y scripts oficiales extraídos;
- reconstruye índices de imágenes y GT;
- perfila valores reales de máscaras PNG;
- intenta pairing por metadata oficial (`Slices Numbers.csv`, `T1_Subfolders.csv`, `T2_Subfolders.csv`, etc.);
- genera overlays por valor/clase para revisión manual;
- produce una decisión técnica trazable.


In [1]:

import importlib.util, subprocess, sys
def ensure_package(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or import_name])

ensure_package("pydicom")
ensure_package("skimage", "scikit-image")

import json, re, zipfile, math, os
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pydicom
from PIL import Image
from scipy import ndimage
from skimage.transform import resize
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as exc:
    print("Drive no montado automáticamente:", repr(exc))

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 160)


Mounted at /content/drive


In [2]:

PFI_ROOT = Path("/content/drive/MyDrive/PFI_MVP")
ALKAFRI_ROOT = PFI_ROOT / "data" / "AXIAL_ALKAFRI"
EXTRACTED_ROOT = ALKAFRI_ROOT / "extracted"
NESTED_ROOT = EXTRACTED_ROOT / "_nested"

E7_ROOT = PFI_ROOT / "results" / "E7_alkafri_axial_curated_subset"
E8_ROOT = PFI_ROOT / "results" / "E8_alkafri_official_pairing"
FIGURES_ROOT = PFI_ROOT / "figures"
DOCS_ROOT = PFI_ROOT / "docs"

for p in [E8_ROOT, FIGURES_ROOT, DOCS_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("PFI_ROOT:", PFI_ROOT)
print("ALKAFRI_ROOT:", ALKAFRI_ROOT)
print("NESTED_ROOT:", NESTED_ROOT)
print("E8_ROOT:", E8_ROOT)

if not NESTED_ROOT.exists():
    raise FileNotFoundError(f"No existe {NESTED_ROOT}. Primero ejecutar extracción/inventario Al-Kafri.")


PFI_ROOT: /content/drive/MyDrive/PFI_MVP
ALKAFRI_ROOT: /content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI
NESTED_ROOT: /content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested
E8_ROOT: /content/drive/MyDrive/PFI_MVP/results/E8_alkafri_official_pairing


In [3]:

def norm_case(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    m = re.search(r"(\d{1,4})", s)
    if not m:
        return None
    n = int(m.group(1))
    if 1 <= n <= 9999:
        return f"{n:04d}"
    return None

def infer_modality(text):
    s = str(text).upper()
    if re.search(r"(^|[^A-Z0-9])T1([^A-Z0-9]|$)", s) or "T1_" in s or "_T1" in s:
        return "T1"
    if re.search(r"(^|[^A-Z0-9])T2([^A-Z0-9]|$)", s) or "T2_" in s or "_T2" in s:
        return "T2"
    return None

def infer_disc(text):
    s = str(text).upper()
    m = re.search(r"D\s*([345])", s)
    return int(m.group(1)) if m else None

def safe_read_csv(path):
    for enc in ["utf-8", "latin1", "cp1252"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            pass
    return pd.read_csv(path, engine="python", encoding_errors="ignore")

def rel(p):
    try:
        return str(Path(p).relative_to(ALKAFRI_ROOT))
    except Exception:
        return str(p)

def read_docx_text(path):
    try:
        with zipfile.ZipFile(path) as zf:
            xml = zf.read("word/document.xml").decode("utf-8", errors="ignore")
        text = re.sub(r"<[^>]+>", " ", xml)
        text = re.sub(r"\s+", " ", text).strip()
        return text
    except Exception as exc:
        return f"[No se pudo leer docx: {exc!r}]"


In [7]:
# 1) Inventario rápido de documentos, metadata y scripts oficiales
# Evita rglob sobre todo AXIAL_ALKAFRI porque en Drive tarda mucho.

candidate_roots = [
    EXTRACTED_ROOT,
    NESTED_ROOT,
    ALKAFRI_ROOT / "raw_zips",
]

wanted_names = [
    "Slices Numbers.csv",
    "T1_Subfolders.csv",
    "T2_Subfolders.csv",
    "PNG counts.csv",
]

print("Buscando archivos oficiales de forma rápida en:")
for r in candidate_roots:
    print("-", r, "| exists:", r.exists())

found = []
seen = set()

# 1) Buscar nombres exactos primero
for root in candidate_roots:
    if not root.exists():
        continue

    for name in wanted_names:
        matches = list(root.rglob(name))
        for p in matches:
            key = str(p)
            if key not in seen and p.is_file():
                seen.add(key)
                found.append(p)

# 2) Buscar solo extensiones oficiales, pero acotado
#    Limitamos a carpetas donde suelen estar source/docs/metadata.
keywords = [
    "source",
    "code",
    "matlab",
    "ground_truth",
    "label",
    "manual",
    "metadata",
    "csv",
]

for root in candidate_roots:
    if not root.exists():
        continue

    for p in root.rglob("*"):
        if not p.is_file():
            continue

        suffix = p.suffix.lower()
        if suffix not in [".csv", ".docx", ".txt", ".md", ".m", ".mat"]:
            continue

        path_lower = str(p).lower()
        name_lower = p.name.lower()

        # Evitar recorrer/guardar cosas irrelevantes de MRI_Data masivo.
        if (
            suffix in [".csv", ".docx", ".txt", ".md", ".m", ".mat"]
            and any(k in path_lower or k in name_lower for k in keywords)
        ):
            key = str(p)
            if key not in seen:
                seen.add(key)
                found.append(p)

docs = [p for p in found if p.suffix.lower() in [".docx", ".txt", ".md"]]
csvs = [p for p in found if p.suffix.lower() == ".csv"]
matlab = [p for p in found if p.suffix.lower() in [".m", ".mat"]]

inventory_rows = []

for p in docs + csvs + matlab:
    try:
        size_bytes = p.stat().st_size
    except Exception:
        size_bytes = None

    inventory_rows.append({
        "file_path": str(p),
        "relative_path": rel(p),
        "name": p.name,
        "suffix": p.suffix.lower(),
        "size_bytes": size_bytes,
    })

inventory_df = pd.DataFrame(
    inventory_rows,
    columns=["file_path", "relative_path", "name", "suffix", "size_bytes"]
)

if len(inventory_df) > 0:
    inventory_df = inventory_df.sort_values(["suffix", "relative_path"])

inventory_df.to_csv(E8_ROOT / "E8_official_files_inventory.csv", index=False)

print("Total official files found:", len(found))
print("docs:", len(docs), "csvs:", len(csvs), "matlab/mat:", len(matlab))
display(inventory_df.head(120))

# Preview de documentos
doc_rows = []

for p in docs[:30]:
    try:
        if p.suffix.lower() == ".docx":
            txt = read_docx_text(p)
        else:
            txt = p.read_text(errors="ignore")

        doc_rows.append({
            "file_path": str(p),
            "relative_path": rel(p),
            "chars": len(txt),
            "preview": txt[:1500],
        })

    except Exception as exc:
        doc_rows.append({
            "file_path": str(p),
            "relative_path": rel(p),
            "chars": None,
            "preview": f"[ERROR leyendo documento: {repr(exc)}]",
        })

docs_preview_df = pd.DataFrame(
    doc_rows,
    columns=["file_path", "relative_path", "chars", "preview"]
)

docs_preview_df.to_csv(E8_ROOT / "E8_docs_preview.csv", index=False)
display(docs_preview_df.head(20))

# Preview de scripts MATLAB
m_rows = []

for p in [x for x in matlab if x.suffix.lower() == ".m"][:80]:
    try:
        txt = p.read_text(errors="ignore")
        hits = bool(
            re.search(
                r"Slice|slice|D3|D4|D5|Ground|Label|SegNet|T1|T2|png|ima|csv",
                txt,
            )
        )

        m_rows.append({
            "file_path": str(p),
            "relative_path": rel(p),
            "chars": len(txt),
            "keyword_hits": hits,
            "preview": txt[:1200],
        })

    except Exception as exc:
        m_rows.append({
            "file_path": str(p),
            "relative_path": rel(p),
            "chars": None,
            "keyword_hits": False,
            "preview": f"[ERROR leyendo MATLAB: {repr(exc)}]",
        })

matlab_preview_df = pd.DataFrame(
    m_rows,
    columns=[
        "file_path",
        "relative_path",
        "chars",
        "keyword_hits",
        "preview",
    ]
)

matlab_preview_df.to_csv(E8_ROOT / "E8_matlab_scripts_preview.csv", index=False)
display(matlab_preview_df.head(20))

Buscando archivos oficiales de forma rápida en:
- /content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted | exists: True
- /content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested | exists: True
- /content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/raw_zips | exists: True
Total official files found: 107
docs: 15 csvs: 5 matlab/mat: 87


,file_path,relative_path,name,suffix,size_bytes
15,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/source_code/09_Source_Code/03_Extract_Best_Slices/Slices Numbers.csv,extracted/source_code/09_Source_Code/03_Extract_Best_Slices/Slices Numbers.csv,Slices Numbers.csv,.csv,6400
16,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/source_code/09_Source_Code/03_Extract_Best_Slices/T1_Subfolders.csv,extracted/source_code/09_Source_Code/03_Extract_Best_Slices/T1_Subfolders.csv,T1_Subfolders.csv,.csv,6261
17,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/source_code/09_Source_Code/03_Extract_Best_Slices/T2_Subfolders.csv,extracted/source_code/09_Source_Code/03_Extract_Best_Slices/T2_Subfolders.csv,T2_Subfolders.csv,.csv,6618
18,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/source_code/09_Source_Code/04_Manual_Label_Checking/PNG counts.csv,extracted/source_code/09_Source_Code/04_Manual_Label_Checking/PNG counts.csv,PNG counts.csv,.csv,5750
19,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/source_code/09_Source_Code/05_Ground_Truth_Development/misregistration.csv,extracted/source_code/09_Source_Code/05_Ground_Truth_Development/misregistration.csv,misregistration.csv,.csv,30
...,...,...,...,...,...
9,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/source_code/09_Source_Code/04_Manual_Label_Checking/Labeller 05 (after postprocess).txt,extracted/source_code/09_Source_Code/04_Manual_Label_Checking/Labeller 05 (after postprocess).txt,Labeller 05 (after postprocess).txt,.txt,0
10,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/source_code/09_Source_Code/04_Manual_Label_Checking/Labeller 05 (before post process).txt,extracted/source_code/09_Source_Code/04_Manual_Label_Checking/Labeller 05 (before post process).txt,Labeller 05 (before post process).txt,.txt,0
11,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/source_code/09_Source_Code/04_Manual_Label_Checking/Mask-Label-Check-Results (after post-process...,extracted/source_code/09_Source_Code/04_Manual_Label_Checking/Mask-Label-Check-Results (after post-process).txt,Mask-Label-Check-Results (after post-process).txt,.txt,405
12,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/source_code/09_Source_Code/04_Manual_Label_Checking/Mask-Label-Check-Results (before post-proces...,extracted/source_code/09_Source_Code/04_Manual_Label_Checking/Mask-Label-Check-Results (before post-process).txt,Mask-Label-Check-Results (before post-process).txt,.txt,405


,file_path,relative_path,chars,preview
0,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/source_code/Instruction.docx,extracted/source_code/Instruction.docx,4534,This document provides a quick instruction on how to use the software code to reproduce the results presented in the paper. Files needed: MRI_Data Manual_La...
1,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/source_code/09_Source_Code/04_Manual_Label_Checking/Labeller 01 (after postprocess).txt,extracted/source_code/09_Source_Code/04_Manual_Label_Checking/Labeller 01 (after postprocess).txt,0,
2,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/source_code/09_Source_Code/04_Manual_Label_Checking/Labeller 01 (before post process).txt,extracted/source_code/09_Source_Code/04_Manual_Label_Checking/Labeller 01 (before post process).txt,0,
3,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/source_code/09_Source_Code/04_Manual_Label_Checking/Labeller 02 (after postprocess).txt,extracted/source_code/09_Source_Code/04_Manual_Label_Checking/Labeller 02 (after postprocess).txt,0,
4,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/source_code/09_Source_Code/04_Manual_Label_Checking/Labeller 02 (before post process).txt,extracted/source_code/09_Source_Code/04_Manual_Label_Checking/Labeller 02 (before post process).txt,0,
5,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/source_code/09_Source_Code/04_Manual_Label_Checking/Labeller 03 (after postprocess).txt,extracted/source_code/09_Source_Code/04_Manual_Label_Checking/Labeller 03 (after postprocess).txt,0,
6,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/source_code/09_Source_Code/04_Manual_Label_Checking/Labeller 03 (before post process).txt,extracted/source_code/09_Source_Code/04_Manual_Label_Checking/Labeller 03 (before post process).txt,0,
7,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/source_code/09_Source_Code/04_Manual_Label_Checking/Labeller 04 (after postprocess).txt,extracted/source_code/09_Source_Code/04_Manual_Label_Checking/Labeller 04 (after postprocess).txt,0,
8,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/source_code/09_Source_Code/04_Manual_Label_Checking/Labeller 04 (before post process).txt,extracted/source_code/09_Source_Code/04_Manual_Label_Checking/Labeller 04 (before post process).txt,0,
9,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/source_code/09_Source_Code/04_Manual_Label_Checking/Labeller 05 (after postprocess).txt,extracted/source_code/09_Source_Code/04_Manual_Label_Checking/Labeller 05 (after postprocess).txt,0,


,file_path,relative_path,chars,keyword_hits,preview
0,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/source_code/initialise.m,extracted/source_code/initialise.m,5193,True,% COPYRIGHT NOTICE -------------------------------------------------------\n% The LJMU Lumbar Spine MRI Image Segmentation and Evaluation software\n% (c) 20...
1,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/source_code/09_Source_Code/03_Extract_Best_Slices/extract_dicom.m,extracted/source_code/09_Source_Code/03_Extract_Best_Slices/extract_dicom.m,4626,True,"function [done] = extract_dicom(directory, startdepth, prefix, patientID, specifiedfolder, slices, outputdir)\n% COPYRIGHT NOTICE --------------------------..."
2,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/source_code/09_Source_Code/03_Extract_Best_Slices/Extract_T1_Slices.m,extracted/source_code/09_Source_Code/03_Extract_Best_Slices/Extract_T1_Slices.m,3405,True,% COPYRIGHT NOTICE -------------------------------------------------------\n% The LJMU Lumbar Spine MRI Image Segmentation and Evaluation software\n% (c) 20...
3,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/source_code/09_Source_Code/03_Extract_Best_Slices/Extract_T2_Slices.m,extracted/source_code/09_Source_Code/03_Extract_Best_Slices/Extract_T2_Slices.m,3212,True,% COPYRIGHT NOTICE -------------------------------------------------------\n% The LJMU Lumbar Spine MRI Image Segmentation and Evaluation software\n% (c) 20...
4,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/source_code/09_Source_Code/03_Extract_Best_Slices/Step_01_Extract_Slices.m,extracted/source_code/09_Source_Code/03_Extract_Best_Slices/Step_01_Extract_Slices.m,1549,True,% COPYRIGHT NOTICE -------------------------------------------------------\n% The LJMU Lumbar Spine MRI Image Segmentation and Evaluation software\n% (c) 20...
5,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/source_code/09_Source_Code/04_Manual_Label_Checking/calc_pf.m,extracted/source_code/09_Source_Code/04_Manual_Label_Checking/calc_pf.m,2620,True,"function [intersct_, union_] = calc_pf(manualLabelFolder, rawLabelFolder)\n% COPYRIGHT NOTICE -------------------------------------------------------\n% The..."
6,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/source_code/09_Source_Code/04_Manual_Label_Checking/check_mask_label.m,extracted/source_code/09_Source_Code/04_Manual_Label_Checking/check_mask_label.m,4071,True,"function [numberOfImageFiles, problemFiles] = check_mask_label(manualmaskdatafolder,foldername)\n% COPYRIGHT NOTICE ----------------------------------------..."
7,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/source_code/09_Source_Code/04_Manual_Label_Checking/count_png_files.m,extracted/source_code/09_Source_Code/04_Manual_Label_Checking/count_png_files.m,2426,True,function [filecounter] = count_png_files(thisFolder)\n% COPYRIGHT NOTICE -------------------------------------------------------\n% The LJMU Lumbar Spine MR...
8,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/source_code/09_Source_Code/04_Manual_Label_Checking/count_votes.m,extracted/source_code/09_Source_Code/04_Manual_Label_Checking/count_votes.m,3938,True,"function [newallvotes, newcounter, newimagesizes] = count_votes(thisFolder, region_labels, allvotes, counter, imagesizes, subfolder)\n% COPYRIGHT NOTICE ---..."
9,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/source_code/09_Source_Code/04_Manual_Label_Checking/get_vote_heatmap.m,extracted/source_code/09_Source_Code/04_Manual_Label_Checking/get_vote_heatmap.m,861,False,"function [croppedlabel] = get_vote_heatmap(allvotes, patientID, disk, region)\n%UNTITLED Summary of this function goes here\n% Detailed explanation goes h..."


In [8]:

# 2) Cargar metadata CSV oficial
metadata_tables = {}
for p in csvs:
    try:
        df = safe_read_csv(p)
        metadata_tables[p.name] = df
        print("\n---", p.name, df.shape, "---")
        print("path:", rel(p))
        display(df.head(10))
    except Exception as exc:
        print("ERROR leyendo", p, repr(exc))

metadata_summary_df = pd.DataFrame([{
    "name": name,
    "rows": len(df),
    "columns": list(df.columns),
} for name, df in metadata_tables.items()])
metadata_summary_df.to_csv(E8_ROOT / "E8_metadata_csv_summary.csv", index=False)
display(metadata_summary_df)



--- Slices Numbers.csv (514, 4) ---
path: extracted/source_code/09_Source_Code/03_Extract_Best_Slices/Slices Numbers.csv


,1,3,7,11
0,2,3,8,13
1,3,3,8,13
2,4,3,8,13
3,5,3,8,13
4,6,3,8,13
5,7,11,14,18
6,8,3,7,10
7,9,3,7,11
8,10,3,8,13
9,11,6,11,14



--- T1_Subfolders.csv (514, 1) ---
path: extracted/source_code/09_Source_Code/03_Extract_Best_Slices/T1_Subfolders.csv


,T1_TSE_TRA
0,T1_TSE_TRA
1,T1_TSE_TRA
2,T1_TSE_TRA
3,T1_TSE_TRA
4,T1_TSE_TRA
5,T1_TSE_TRA
6,T1_TSE_TRA
7,T1_TSE_TRA
8,T1_TSE_TRA
9,T1_TSE_TRA



--- T2_Subfolders.csv (514, 1) ---
path: extracted/source_code/09_Source_Code/03_Extract_Best_Slices/T2_Subfolders.csv


...


--- PNG counts.csv (574, 5) ---
path: extracted/source_code/09_Source_Code/04_Manual_Label_Checking/PNG counts.csv


,3,3.1,3.2,3.3,3.4
0,3,3,3,3,3
1,3,3,3,3,3
2,3,3,3,3,3
3,3,3,3,3,3
4,3,3,3,3,3
5,3,3,3,3,3
6,3,3,3,3,3
7,3,3,3,3,3
8,3,3,3,3,3
9,3,3,3,3,3



--- misregistration.csv (9, 2) ---
path: extracted/source_code/09_Source_Code/05_Ground_Truth_Development/misregistration.csv


,Unnamed: 0,Unnamed: 1
0,NaN,NaN
1,NaN,NaN
2,NaN,NaN
3,NaN,NaN
4,NaN,NaN
5,NaN,NaN
6,NaN,NaN
7,NaN,NaN
8,NaN,NaN


,name,rows,columns
0,Slices Numbers.csv,514,"[1, 3, 7, 11]"
1,T1_Subfolders.csv,514,[T1_TSE_TRA]
2,T2_Subfolders.csv,514,[T2_TSE_TRA]
3,PNG counts.csv,574,"[3, 3.1, 3.2, 3.3, 3.4]"
4,misregistration.csv,9,"[Unnamed: 0, Unnamed: 1]"


In [15]:

# 3) Cargar índices de imágenes axiales y GT generados en E7
image_index_path = E7_ROOT / "E7_alkafri_axial_image_case_index.csv"
gt_index_path = E7_ROOT / "E7_alkafri_gt_case_index.csv"

if image_index_path.exists() and gt_index_path.exists():
    image_df = pd.read_csv(image_index_path, low_memory=False)
    gt_df = pd.read_csv(gt_index_path, low_memory=False)
    print("Usando índices E7 existentes.")
else:
    raise FileNotFoundError("Faltan índices E7. Ejecutar notebook 15/patch de indexación antes de este notebook.")

image_df["case_id_norm"] = image_df["case_id"].apply(norm_case)
gt_df["case_id_norm"] = gt_df["case_id"].apply(norm_case)
image_df["modality"] = image_df["modality"].fillna(image_df["image_relative_path"].apply(infer_modality))
gt_df["modality"] = gt_df["modality"].fillna(gt_df["gt_relative_path"].apply(infer_modality))
gt_df["disc_or_slice_id"] = gt_df["disc_or_slice_id"].fillna(gt_df["gt_relative_path"].apply(infer_disc))

print("image_df:", image_df.shape)
print("gt_df:", gt_df.shape)
display(image_df.head())
display(gt_df.head())

image_df.to_csv(E8_ROOT / "E8_image_index_normalized.csv", index=False)
gt_df.to_csv(E8_ROOT / "E8_gt_index_normalized.csv", index=False)


Usando índices E7 existentes.
image_df: (8150, 9)
gt_df: (37080, 9)


,image_file_path,image_relative_path,case_id,modality,series_id,series_description,instance_number,is_posdisp_or_localizer,case_id_norm
0,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0037/L-SPINE_LSS_20150919_125616_225000/T2_TSE_TRA_38...,_nested/main_dataset__MRI_Data/01_MRI_Data/0037/L-SPINE_LSS_20150919_125616_225000/T2_TSE_TRA_384_0004/T2_TSE_TRA__0037_001.ima,37,T2,1.3.12.2.1107.5.2.40.50233.2015091913064063223406200.0.0.0,t2_tse_tra_384,1,False,0037
1,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0037/L-SPINE_LSS_20150919_125616_225000/T2_TSE_TRA_38...,_nested/main_dataset__MRI_Data/01_MRI_Data/0037/L-SPINE_LSS_20150919_125616_225000/T2_TSE_TRA_384_0004/T2_TSE_TRA__0037_002.ima,37,T2,1.3.12.2.1107.5.2.40.50233.2015091913064063223406200.0.0.0,t2_tse_tra_384,2,False,0037
2,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0037/L-SPINE_LSS_20150919_125616_225000/T2_TSE_TRA_38...,_nested/main_dataset__MRI_Data/01_MRI_Data/0037/L-SPINE_LSS_20150919_125616_225000/T2_TSE_TRA_384_0004/T2_TSE_TRA__0037_003.ima,37,T2,1.3.12.2.1107.5.2.40.50233.2015091913064063223406200.0.0.0,t2_tse_tra_384,3,False,0037
3,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0037/L-SPINE_LSS_20150919_125616_225000/T2_TSE_TRA_38...,_nested/main_dataset__MRI_Data/01_MRI_Data/0037/L-SPINE_LSS_20150919_125616_225000/T2_TSE_TRA_384_0004/T2_TSE_TRA__0037_004.ima,37,T2,1.3.12.2.1107.5.2.40.50233.2015091913064063223406200.0.0.0,t2_tse_tra_384,4,False,0037
4,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0037/L-SPINE_LSS_20150919_125616_225000/T2_TSE_TRA_38...,_nested/main_dataset__MRI_Data/01_MRI_Data/0037/L-SPINE_LSS_20150919_125616_225000/T2_TSE_TRA_384_0004/T2_TSE_TRA__0037_005.ima,37,T2,1.3.12.2.1107.5.2.40.50233.2015091913064063223406200.0.0.0,t2_tse_tra_384,5,False,0037


,gt_file_path,gt_relative_path,case_id,modality,disc_or_slice_id,labeller,gt_type,extension,case_id_norm
0,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Manual_Label_Data/03_Manual_Label_Data/Labeller 01/T1_0001_D3.png,_nested/ground_truth__Manual_Label_Data/03_Manual_Label_Data/Labeller 01/T1_0001_D3.png,1,T1,3,1.0,manual,.png,0001
1,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Manual_Label_Data/03_Manual_Label_Data/Labeller 01/T1_0001_D3.xcf,_nested/ground_truth__Manual_Label_Data/03_Manual_Label_Data/Labeller 01/T1_0001_D3.xcf,1,T1,3,1.0,manual,.xcf,0001
2,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Manual_Label_Data/03_Manual_Label_Data/Labeller 01/T1_0001_D4.png,_nested/ground_truth__Manual_Label_Data/03_Manual_Label_Data/Labeller 01/T1_0001_D4.png,1,T1,4,1.0,manual,.png,0001
3,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Manual_Label_Data/03_Manual_Label_Data/Labeller 01/T1_0001_D4.xcf,_nested/ground_truth__Manual_Label_Data/03_Manual_Label_Data/Labeller 01/T1_0001_D4.xcf,1,T1,4,1.0,manual,.xcf,0001
4,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Manual_Label_Data/03_Manual_Label_Data/Labeller 01/T1_0001_D5.png,_nested/ground_truth__Manual_Label_Data/03_Manual_Label_Data/Labeller 01/T1_0001_D5.png,1,T1,5,1.0,manual,.png,0001


In [16]:
# PATCH E8: reconstrucción oficial correcta usando CSVs sin header
# Ejecutar después de tener cargados:
# - metadata_tables
# - image_df
# - gt_df
# - csvs
# - E8_ROOT

def find_official_csv(name):
    # Primero buscar en la lista csvs si existe
    for p in csvs:
        if p.name == name:
            return p

    # Fallback por si csvs no está completo
    matches = list(ALKAFRI_ROOT.rglob(name))
    if not matches:
        raise FileNotFoundError(f"No encontré {name}")
    return matches[0]

def norm_text_for_match(x):
    s = str(x).lower()
    s = re.sub(r"[^a-z0-9]+", "", s)
    return s

slices_csv = find_official_csv("Slices Numbers.csv")
t1_csv = find_official_csv("T1_Subfolders.csv")
t2_csv = find_official_csv("T2_Subfolders.csv")

# Muy importante: header=None.
slices_raw = pd.read_csv(slices_csv, header=None)
t1_sub_raw = pd.read_csv(t1_csv, header=None)
t2_sub_raw = pd.read_csv(t2_csv, header=None)

print("slices_raw:", slices_raw.shape)
print("t1_sub_raw:", t1_sub_raw.shape)
print("t2_sub_raw:", t2_sub_raw.shape)

display(slices_raw.head())
display(t1_sub_raw.head())
display(t2_sub_raw.head())

# Esperado:
# slices_raw columnas:
# 0 = case_id / patient_id
# 1 = slice asociado a D3
# 2 = slice asociado a D4
# 3 = slice asociado a D5

slices_raw = slices_raw.iloc[:, :4].copy()
slices_raw.columns = ["case_num", "D3_slice", "D4_slice", "D5_slice"]

t1_sub_raw = t1_sub_raw.iloc[:, :1].copy()
t1_sub_raw.columns = ["T1_subfolder"]

t2_sub_raw = t2_sub_raw.iloc[:, :1].copy()
t2_sub_raw.columns = ["T2_subfolder"]

n = min(len(slices_raw), len(t1_sub_raw), len(t2_sub_raw))
slices_raw = slices_raw.iloc[:n].copy()
t1_sub_raw = t1_sub_raw.iloc[:n].copy()
t2_sub_raw = t2_sub_raw.iloc[:n].copy()

official_rows = []

for idx in range(n):
    case_num = slices_raw.loc[idx, "case_num"]
    case_id_norm = norm_case(case_num)

    if case_id_norm is None:
        continue

    t1_subfolder = str(t1_sub_raw.loc[idx, "T1_subfolder"])
    t2_subfolder = str(t2_sub_raw.loc[idx, "T2_subfolder"])

    for disc_id, col in [(3, "D3_slice"), (4, "D4_slice"), (5, "D5_slice")]:
        slice_number = pd.to_numeric(slices_raw.loc[idx, col], errors="coerce")

        if pd.isna(slice_number):
            continue

        slice_number = int(slice_number)

        for modality, subfolder in [
            ("T1", t1_subfolder),
            ("T2", t2_subfolder),
        ]:
            official_rows.append({
                "case_id_norm": case_id_norm,
                "case_num": int(case_num),
                "modality": modality,
                "disc_id": disc_id,
                "slice_number": slice_number,
                "subfolder": subfolder,
                "subfolder_norm": norm_text_for_match(subfolder),
                "source": "official_csv_header_none",
            })

official_slice_df = pd.DataFrame(official_rows)

official_slice_df.to_csv(
    E8_ROOT / "E8_official_slice_metadata_header_none.csv",
    index=False,
)

print("official_slice_df:", official_slice_df.shape)
display(official_slice_df.head(20))
display(
    official_slice_df
    .groupby(["modality", "disc_id"], dropna=False)
    .size()
    .reset_index(name="n")
)

# Preparar imágenes
img = image_df.copy()
img["case_id_norm"] = img["case_id"].apply(norm_case)
img["instance_number_num"] = pd.to_numeric(
    img["instance_number"],
    errors="coerce"
).astype("Int64")

img["series_description_norm"] = img["series_description"].apply(norm_text_for_match)
img["image_relative_path_norm"] = img["image_relative_path"].apply(norm_text_for_match)

# Preparar GT PNG
gt_png = gt_df[
    gt_df["extension"].astype(str).str.lower().eq(".png")
].copy()

gt_png["case_id_norm"] = gt_png["case_id"].apply(norm_case)
gt_png["disc_id"] = pd.to_numeric(
    gt_png["disc_or_slice_id"],
    errors="coerce"
).astype("Int64")

# Para entrenamiento futuro priorizamos final/intermediary.
# Manual sirve para diagnóstico, pero no lo tomaría de entrada como target principal.
gt_png = gt_png[
    gt_png["gt_type"].isin(["final", "intermediary", "manual"])
].copy()

print("img:", img.shape)
print("gt_png:", gt_png.shape)

# Matching por slice oficial.
# Probamos offset 0, -1, +1 por si hay diferencia de indexación.
candidate_img_rows = []

for offset in [0, -1, 1]:
    sx = official_slice_df.copy()
    sx["match_instance_number"] = sx["slice_number"] + offset
    sx["slice_offset"] = offset

    merged = img.merge(
        sx,
        left_on=["case_id_norm", "modality", "instance_number_num"],
        right_on=["case_id_norm", "modality", "match_instance_number"],
        how="inner",
        suffixes=("_img", "_official"),
    )

    if len(merged) == 0:
        continue

    # Confirmar subfolder oficial contra ruta o descripción.
    merged["series_match"] = merged.apply(
        lambda r: (
            r["subfolder_norm"] in r["image_relative_path_norm"]
            or r["subfolder_norm"] in r["series_description_norm"]
            or r["series_description_norm"] in r["subfolder_norm"]
        ),
        axis=1,
    )

    # Primero guardar los que matchean por subfolder.
    candidate_img_rows.append(merged[merged["series_match"]].copy())

candidate_img_df = (
    pd.concat(candidate_img_rows, ignore_index=True)
    if candidate_img_rows
    else pd.DataFrame()
)

print("candidate_img_df:", candidate_img_df.shape)

if len(candidate_img_df) == 0:
    print("No hubo match con subfolder estricto. Probando match relajado sin subfolder...")

    candidate_img_rows = []

    for offset in [0, -1, 1]:
        sx = official_slice_df.copy()
        sx["match_instance_number"] = sx["slice_number"] + offset
        sx["slice_offset"] = offset

        merged = img.merge(
            sx,
            left_on=["case_id_norm", "modality", "instance_number_num"],
            right_on=["case_id_norm", "modality", "match_instance_number"],
            how="inner",
            suffixes=("_img", "_official"),
        )

        candidate_img_rows.append(merged.copy())

    candidate_img_df = (
        pd.concat(candidate_img_rows, ignore_index=True)
        if candidate_img_rows
        else pd.DataFrame()
    )

print("candidate_img_df final:", candidate_img_df.shape)

if len(candidate_img_df) > 0:
    display(
        candidate_img_df[
            [
                "case_id_norm",
                "modality",
                "disc_id",
                "slice_number",
                "slice_offset",
                "match_instance_number",
                "instance_number_num",
                "subfolder",
                "series_description",
                "series_match" if "series_match" in candidate_img_df.columns else "image_relative_path",
            ]
        ].head(30)
    )

# Join con GT por caso, modalidad y disco.
if len(candidate_img_df) > 0:
    official_candidates = candidate_img_df.merge(
        gt_png,
        left_on=["case_id_norm", "modality", "disc_id"],
        right_on=["case_id_norm", "modality", "disc_id"],
        how="inner",
        suffixes=("_img", "_gt"),
    )
else:
    official_candidates = pd.DataFrame()

print("official_candidates merged:", official_candidates.shape)

# Priorizar:
# 1) offset 0
# 2) gt final
# 3) gt intermediary
# 4) gt manual
if len(official_candidates) > 0:
    gt_priority_map = {
        "final": 0,
        "intermediary": 1,
        "manual": 2,
    }

    official_candidates["gt_priority"] = official_candidates["gt_type"].map(
        gt_priority_map
    ).fillna(9)

    official_candidates["offset_abs"] = official_candidates["slice_offset"].abs()

    official_candidates = official_candidates.sort_values(
        [
            "offset_abs",
            "gt_priority",
            "case_id_norm",
            "modality",
            "disc_id",
            "instance_number_num",
        ]
    )

    # Evitar duplicados exactos imagen-GT.
    official_candidates = official_candidates.drop_duplicates(
        subset=["image_file_path", "gt_file_path"]
    )

    official_candidates_df = pd.DataFrame({
        "strategy": "official_slice_number_header_none",
        "case_id": official_candidates["case_id_norm"],
        "modality": official_candidates["modality"],
        "disc_id": official_candidates["disc_id"].astype(int),
        "slice_number": official_candidates["slice_number"].astype(int),
        "slice_offset": official_candidates["slice_offset"].astype(int),
        "instance_number": official_candidates["instance_number_num"].astype(int),
        "subfolder": official_candidates["subfolder"],
        "series_description": official_candidates["series_description"],
        "image_file_path": official_candidates["image_file_path"],
        "gt_file_path": official_candidates["gt_file_path"],
        "gt_type": official_candidates["gt_type"],
        "evidence": (
            "official Slices Numbers.csv + "
            "T1/T2_Subfolders.csv + "
            "case/modality/disc/slice match"
        ),
    })

else:
    official_candidates_df = pd.DataFrame(
        columns=[
            "strategy",
            "case_id",
            "modality",
            "disc_id",
            "slice_number",
            "slice_offset",
            "instance_number",
            "subfolder",
            "series_description",
            "image_file_path",
            "gt_file_path",
            "gt_type",
            "evidence",
        ]
    )

official_candidates_df.to_csv(
    E8_ROOT / "E8_official_pairing_candidates_HEADER_NONE.csv",
    index=False,
)

print("official_candidates_df:", official_candidates_df.shape)

if len(official_candidates_df) > 0:
    display(official_candidates_df.head(50))
    display(
        official_candidates_df
        .groupby(["strategy", "modality", "gt_type", "slice_offset"], dropna=False)
        .size()
        .reset_index(name="n")
        .sort_values("n", ascending=False)
    )
else:
    print("No se generaron candidatos oficiales. Habría que inspeccionar offset/series/manual.")

slices_raw: (515, 4)
t1_sub_raw: (515, 1)
t2_sub_raw: (515, 1)


,0,1,2,3
0,1,3,7,11
1,2,3,8,13
2,3,3,8,13
3,4,3,8,13
4,5,3,8,13


,0
0,T1_TSE_TRA
1,T1_TSE_TRA
2,T1_TSE_TRA
3,T1_TSE_TRA
4,T1_TSE_TRA


,0
0,T2_TSE_TRA
1,T2_TSE_TRA
2,T2_TSE_TRA
3,T2_TSE_TRA
4,T2_TSE_TRA


official_slice_df: (3090, 8)


...

img: (8150, 12)
gt_png: (29355, 10)
candidate_img_df: (3959, 21)
candidate_img_df final: (3959, 21)


,case_id_norm,modality,disc_id,slice_number,slice_offset,match_instance_number,instance_number_num,subfolder,series_description,series_match
0,0037,T2,3,7,0,7,7,T2_TSE_TRA,t2_tse_tra_384,True
1,0037,T2,4,11,0,11,11,T2_TSE_TRA,t2_tse_tra_384,True
2,0037,T2,5,14,0,14,14,T2_TSE_TRA,t2_tse_tra_384,True
3,0037,T1,3,7,0,7,7,T1_TSE_TRA,t1_tse_tra,True
4,0037,T1,4,11,0,11,11,T1_TSE_TRA,t1_tse_tra,True
5,0037,T1,5,14,0,14,14,T1_TSE_TRA,t1_tse_tra,True
6,0198,T2,3,3,0,3,3,T2_TSE_TRA,t2_tse_tra_384,True
7,0198,T2,3,3,0,3,3,T2_TSE_TRA,t2_tse_tra_384,True
8,0198,T2,4,8,0,8,8,T2_TSE_TRA,t2_tse_tra_384,True
9,0198,T2,4,8,0,8,8,T2_TSE_TRA,t2_tse_tra_384,True


official_candidates merged: (22079, 28)
official_candidates_df: (22079, 13)


,strategy,case_id,modality,disc_id,slice_number,slice_offset,instance_number,subfolder,series_description,image_file_path,gt_file_path,gt_type,evidence
3914,official_slice_number_header_none,0001,T1,3,3,0,3,T1_TSE_TRA,t1_tse_tra,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0001/L-SPINE_LSS_20160309_091629_240000/T1_TSE_TRA_00...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T1_Output/T1_0001_D3.png,intermediary,official Slices Numbers.csv + T1/T2_Subfolders.csv + case/modality/disc/slice match
3925,official_slice_number_header_none,0001,T1,4,7,0,7,T1_TSE_TRA,t1_tse_tra,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0001/L-SPINE_LSS_20160309_091629_240000/T1_TSE_TRA_00...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T1_Output/T1_0001_D4.png,intermediary,official Slices Numbers.csv + T1/T2_Subfolders.csv + case/modality/disc/slice match
3936,official_slice_number_header_none,0001,T1,5,11,0,11,T1_TSE_TRA,t1_tse_tra,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0001/L-SPINE_LSS_20160309_091629_240000/T1_TSE_TRA_00...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T1_Output/T1_0001_D5.png,intermediary,official Slices Numbers.csv + T1/T2_Subfolders.csv + case/modality/disc/slice match
3901,official_slice_number_header_none,0001,T2,3,3,0,3,T2_TSE_TRA,t2_tse_tra_384,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0001/L-SPINE_LSS_20160309_091629_240000/T2_TSE_TRA_38...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T2_Output/T2_0001_D3.png,intermediary,official Slices Numbers.csv + T1/T2_Subfolders.csv + case/modality/disc/slice match
3902,official_slice_number_header_none,0001,T2,4,7,0,7,T2_TSE_TRA,t2_tse_tra_384,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0001/L-SPINE_LSS_20160309_091629_240000/T2_TSE_TRA_38...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T2_Output/T2_0001_D4.png,intermediary,official Slices Numbers.csv + T1/T2_Subfolders.csv + case/modality/disc/slice match
7220,official_slice_number_header_none,0001,T2,4,7,0,7,T2_TSE_TRA,PosDisp: [4] t2_tse_tra_384,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0001/L-SPINE_LSS_20160309_091629_240000/POSDISP_[4]_T...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T2_Output/T2_0001_D4.png,intermediary,official Slices Numbers.csv + T1/T2_Subfolders.csv + case/modality/disc/slice match
7221,official_slice_number_header_none,0001,T2,4,7,0,7,T2_TSE_TRA,PosDisp: [4] t2_tse_tra_384,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0001/L-SPINE_LSS_20160309_091629_240000/POSDISP_[4]_T...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T2_Output/T2_0001_D4.png,intermediary,official Slices Numbers.csv + T1/T2_Subfolders.csv + case/modality/disc/slice match
3903,official_slice_number_header_none,0001,T2,5,11,0,11,T2_TSE_TRA,t2_tse_tra_384,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0001/L-SPINE_LSS_20160309_091629_240000/T2_TSE_TRA_38...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04

,strategy,modality,gt_type,slice_offset,n
4,official_slice_number_header_none,T1,manual,0,6070
3,official_slice_number_header_none,T1,manual,-1,6060
5,official_slice_number_header_none,T1,manual,1,5990
6,official_slice_number_header_none,T2,intermediary,-1,778
7,official_slice_number_header_none,T2,intermediary,0,717
8,official_slice_number_header_none,T2,intermediary,1,652
1,official_slice_number_header_none,T1,intermediary,0,607
0,official_slice_number_header_none,T1,intermediary,-1,606
2,official_slice_number_header_none,T1,intermediary,1,599


In [23]:
# RESTORE + FILTER de candidatos oficiales correctos
# Ejecutar después del patch HEADER_NONE y antes de visualizar overlays.

official_candidates_path = E8_ROOT / "E8_official_pairing_candidates_HEADER_NONE.csv"

official_candidates_df = pd.read_csv(official_candidates_path)

print("Candidatos oficiales crudos:", official_candidates_df.shape)

# Asegurar tipos
official_candidates_df["slice_offset"] = pd.to_numeric(
    official_candidates_df["slice_offset"],
    errors="coerce"
)

official_candidates_df["instance_number"] = pd.to_numeric(
    official_candidates_df["instance_number"],
    errors="coerce"
)

official_candidates_df["disc_id"] = pd.to_numeric(
    official_candidates_df["disc_id"],
    errors="coerce"
)

# Filtrado conservador para revisión visual:
# - Solo offset 0.
# - Excluir PosDisp/localizer.
# - Priorizar ground truth consolidado/intermediary, no manual de labellers.
official_candidates_df = official_candidates_df[
    official_candidates_df["slice_offset"].eq(0)
].copy()

official_candidates_df = official_candidates_df[
    ~official_candidates_df["series_description"]
    .astype(str)
    .str.contains("PosDisp|localizer", case=False, na=False)
].copy()

official_candidates_df = official_candidates_df[
    ~official_candidates_df["image_file_path"]
    .astype(str)
    .str.contains("POSDISP|LOCALIZER", case=False, na=False)
].copy()

official_candidates_df = official_candidates_df[
    official_candidates_df["gt_type"].isin(["final", "intermediary"])
].copy()

# Evitar duplicados exactos.
official_candidates_df = official_candidates_df.drop_duplicates(
    subset=[
        "case_id",
        "modality",
        "disc_id",
        "slice_number",
        "instance_number",
        "image_file_path",
        "gt_file_path",
    ]
).copy()

official_candidates_df = official_candidates_df.sort_values(
    [
        "case_id",
        "modality",
        "disc_id",
        "gt_type",
        "instance_number",
    ]
)

official_candidates_df.to_csv(
    E8_ROOT / "E8_official_pairing_candidates_HEADER_NONE_FILTERED.csv",
    index=False,
)

print("Candidatos oficiales filtrados:", official_candidates_df.shape)

display(official_candidates_df.head(30))

display(
    official_candidates_df
    .groupby(["strategy", "modality", "gt_type", "slice_offset"], dropna=False)
    .size()
    .reset_index(name="n")
    .sort_values("n", ascending=False)
)

assert len(official_candidates_df) > 0, "No quedaron candidatos oficiales filtrados."
assert official_candidates_df["strategy"].astype(str).str.contains("official_slice_number_header_none").all()

print("OK: ahora official_candidates_df contiene SOLO candidatos oficiales.")

Candidatos oficiales crudos: (22079, 13)
Candidatos oficiales filtrados: (1201, 13)


,strategy,case_id,modality,disc_id,slice_number,slice_offset,instance_number,subfolder,series_description,image_file_path,gt_file_path,gt_type,evidence
0,official_slice_number_header_none,1,T1,3,3,0,3,T1_TSE_TRA,t1_tse_tra,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0001/L-SPINE_LSS_20160309_091629_240000/T1_TSE_TRA_00...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T1_Output/T1_0001_D3.png,intermediary,official Slices Numbers.csv + T1/T2_Subfolders.csv + case/modality/disc/slice match
1,official_slice_number_header_none,1,T1,4,7,0,7,T1_TSE_TRA,t1_tse_tra,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0001/L-SPINE_LSS_20160309_091629_240000/T1_TSE_TRA_00...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T1_Output/T1_0001_D4.png,intermediary,official Slices Numbers.csv + T1/T2_Subfolders.csv + case/modality/disc/slice match
2,official_slice_number_header_none,1,T1,5,11,0,11,T1_TSE_TRA,t1_tse_tra,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0001/L-SPINE_LSS_20160309_091629_240000/T1_TSE_TRA_00...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T1_Output/T1_0001_D5.png,intermediary,official Slices Numbers.csv + T1/T2_Subfolders.csv + case/modality/disc/slice match
3,official_slice_number_header_none,1,T2,3,3,0,3,T2_TSE_TRA,t2_tse_tra_384,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0001/L-SPINE_LSS_20160309_091629_240000/T2_TSE_TRA_38...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T2_Output/T2_0001_D3.png,intermediary,official Slices Numbers.csv + T1/T2_Subfolders.csv + case/modality/disc/slice match
4,official_slice_number_header_none,1,T2,4,7,0,7,T2_TSE_TRA,t2_tse_tra_384,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0001/L-SPINE_LSS_20160309_091629_240000/T2_TSE_TRA_38...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T2_Output/T2_0001_D4.png,intermediary,official Slices Numbers.csv + T1/T2_Subfolders.csv + case/modality/disc/slice match
7,official_slice_number_header_none,1,T2,5,11,0,11,T2_TSE_TRA,t2_tse_tra_384,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0001/L-SPINE_LSS_20160309_091629_240000/T2_TSE_TRA_38...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T2_Output/T2_0001_D5.png,intermediary,official Slices Numbers.csv + T1/T2_Subfolders.csv + case/modality/disc/slice match
8,official_slice_number_header_none,2,T1,3,3,0,3,T1_TSE_TRA,t1_tse_tra,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0002/L-SPINE_CLINICAL_LIBRARIES_20160621_112938_87300...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T1_Output/T1_0002_D3.png,intermediary,official Slices Numbers.csv + T1/T2_Subfolders.csv + case/modality/disc/slice match
9,official_slice_number_header_none,2,T1,4,8,0,8,T1_TSE_TRA,t1_tse_tra,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0002/L-SPINE_CLINICAL_LIBRARIES_20160621_112938_87300...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T1_Output/T1_0002_D4.png,intermediary,official Sli

,strategy,modality,gt_type,slice_offset,n
1,official_slice_number_header_none,T2,intermediary,0,610
0,official_slice_number_header_none,T1,intermediary,0,591


OK: ahora official_candidates_df contiene SOLO candidatos oficiales.


In [24]:

# 4) Parsear Slices Numbers.csv a formato largo, de forma robusta
def parse_slices_metadata(metadata_tables):
    rows = []
    for table_name, df in metadata_tables.items():
        if "slice" not in table_name.lower() and "slices" not in table_name.lower():
            continue
        for ridx, row in df.iterrows():
            row_text = " ".join([str(v) for v in row.values if pd.notna(v)])
            case_id = None
            modality = infer_modality(row_text)
            disc = infer_disc(row_text)
            for col in df.columns:
                if re.search(r"case|patient|subject|id", str(col), re.I):
                    case_id = norm_case(row[col])
                    if case_id:
                        break
            if not case_id:
                case_id = norm_case(row_text)

            # Columnas D3/D4/D5 explícitas
            has_disc_cols = False
            for col, v in row.items():
                d_col = infer_disc(col)
                if d_col and pd.notna(v):
                    has_disc_cols = True
                    for m in re.finditer(r"\b\d{1,2}\b", str(v)):
                        n = int(m.group(0))
                        if 1 <= n <= 60:
                            rows.append({
                                "source_table": table_name,
                                "source_row": ridx,
                                "case_id": case_id,
                                "modality": modality,
                                "disc_id": d_col,
                                "slice_number": n,
                                "source_column": str(col),
                                "source_value": str(v),
                            })
            # Fallback si no hay columnas D explícitas
            if not has_disc_cols:
                for col, v in row.items():
                    if pd.isna(v):
                        continue
                    d_val = infer_disc(v) or disc
                    for m in re.finditer(r"\b\d{1,2}\b", str(v)):
                        n = int(m.group(0))
                        if 1 <= n <= 60:
                            rows.append({
                                "source_table": table_name,
                                "source_row": ridx,
                                "case_id": case_id,
                                "modality": modality,
                                "disc_id": d_val,
                                "slice_number": n,
                                "source_column": str(col),
                                "source_value": str(v),
                            })
    return pd.DataFrame(rows).drop_duplicates() if rows else pd.DataFrame()

slices_long_df = parse_slices_metadata(metadata_tables)
slices_long_df.to_csv(E8_ROOT / "E8_slices_numbers_long_candidate.csv", index=False)
print("slices_long_df:", slices_long_df.shape)
display(slices_long_df.head(50))
if len(slices_long_df):
    display(slices_long_df.groupby(["source_table"], dropna=False).size().reset_index(name="n"))
    display(slices_long_df.groupby(["modality", "disc_id"], dropna=False).size().reset_index(name="n"))


slices_long_df: (1595, 8)


,source_table,source_row,case_id,modality,disc_id,slice_number,source_column,source_value
0,Slices Numbers.csv,0,0002,None,None,2,1,2
1,Slices Numbers.csv,0,0002,None,None,3,3,3
2,Slices Numbers.csv,0,0002,None,None,8,7,8
3,Slices Numbers.csv,0,0002,None,None,13,11,13
4,Slices Numbers.csv,1,0003,None,None,3,1,3
5,Slices Numbers.csv,1,0003,None,None,3,3,3
6,Slices Numbers.csv,1,0003,None,None,8,7,8
7,Slices Numbers.csv,1,0003,None,None,13,11,13
8,Slices Numbers.csv,2,0004,None,None,4,1,4
9,Slices Numbers.csv,2,0004,None,None,3,3,3


,source_table,n
0,Slices Numbers.csv,1595


,modality,disc_id,n
0,NaN,NaN,1595


In [25]:

# 5) Perfil de valores reales de máscaras PNG
def read_mask_raw(path):
    with Image.open(path) as img:
        arr = np.asarray(img)
    if arr.ndim == 3 and arr.shape[-1] == 4:
        arr = arr[..., :3]
    return arr

def value_profile(mask_path, max_values=30):
    arr = read_mask_raw(mask_path)
    if arr.ndim == 3:
        flat = arr.reshape(-1, arr.shape[-1])
        vals, counts = np.unique(flat, axis=0, return_counts=True)
        values = [tuple(map(int, v.tolist())) for v in vals]
    else:
        vals, counts = np.unique(arr.reshape(-1), return_counts=True)
        values = [int(v) for v in vals]
    total = int(np.prod(arr.shape[:2]))
    prof = []
    order = np.argsort(counts)[::-1]
    for idx in order[:max_values]:
        val = values[idx]
        cnt = int(counts[idx])
        prof.append({"value": str(val), "count": cnt, "ratio": cnt / total})
    return arr.shape, prof

gt_png = gt_df[
    gt_df["extension"].astype(str).str.lower().eq(".png")
    & gt_df["case_id_norm"].notna()
    & gt_df["modality"].notna()
].copy()

sample_gt = (
    gt_png.sort_values(["gt_type", "modality", "case_id_norm", "disc_or_slice_id"])
    .groupby(["gt_type", "modality"], group_keys=False)
    .head(25)
    .copy()
)

profile_rows = []
for _, row in tqdm(sample_gt.iterrows(), total=len(sample_gt), desc="mask value profiles"):
    try:
        shape, prof = value_profile(row["gt_file_path"])
        for item in prof:
            profile_rows.append({
                "gt_file_path": row["gt_file_path"],
                "gt_relative_path": row["gt_relative_path"],
                "case_id": row["case_id_norm"],
                "modality": row["modality"],
                "gt_type": row["gt_type"],
                "disc_or_slice_id": row["disc_or_slice_id"],
                "raw_shape": str(shape),
                **item
            })
    except Exception as exc:
        profile_rows.append({"gt_file_path": row["gt_file_path"], "error": repr(exc)})

mask_value_profile_df = pd.DataFrame(profile_rows)
mask_value_profile_df.to_csv(E8_ROOT / "E8_mask_value_profile_sample.csv", index=False)
display(mask_value_profile_df.head(80))
display(mask_value_profile_df.groupby(["gt_type", "modality", "value"], dropna=False)["ratio"].median().reset_index().sort_values(["gt_type", "modality", "ratio"], ascending=[True, True, False]).head(80))


mask value profiles:   0%|          | 0/75 [00:00<?, ?it/s]

,gt_file_path,gt_relative_path,case_id,modality,gt_type,disc_or_slice_id,raw_shape,value,count,ratio
0,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T1_Output/T1_0001_D3.png,_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T1_Output/T1_0001_D3.png,0001,T1,intermediary,3,"(320, 320)",9,3896,0.038047
1,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T1_Output/T1_0001_D3.png,_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T1_Output/T1_0001_D3.png,0001,T1,intermediary,3,"(320, 320)",8,3841,0.037510
2,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T1_Output/T1_0001_D3.png,_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T1_Output/T1_0001_D3.png,0001,T1,intermediary,3,"(320, 320)",10,3590,0.035059
3,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T1_Output/T1_0001_D3.png,_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T1_Output/T1_0001_D3.png,0001,T1,intermediary,3,"(320, 320)",11,3279,0.032021
4,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T1_Output/T1_0001_D3.png,_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T1_Output/T1_0001_D3.png,0001,T1,intermediary,3,"(320, 320)",6,3111,0.030381
...,...,...,...,...,...,...,...,...,...,...
75,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T1_Output/T1_0001_D5.png,_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T1_Output/T1_0001_D5.png,0001,T1,intermediary,5,"(320, 320)",29,906,0.008848
76,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T1_Output/T1_0001_D5.png,_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T1_Output/T1_0001_D5.png,0001,T1,intermediary,5,"(320, 320)",36,897,0.008760
77,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T1_Output/T1_0001_D5.png,_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T1_Output/T1_0001_D5.png,0001,T1,intermediary,5,"(320, 320)",37,890,0.008691
78,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T1_Output/T1_0001_D5.png,_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T1_Output/T1_0001_D5.png,0001,T1,intermediary,5,"(320, 320)",38,877,0.008564


,gt_type,modality,value,ratio
12,intermediary,T1,2,0.059697
23,intermediary,T1,3,0.056211
34,intermediary,T1,4,0.052832
45,intermediary,T1,5,0.041357
0,intermediary,T1,0,0.036250
...,...,...,...,...
67,intermediary,T2,14,0.017480
68,intermediary,T2,15,0.016602
69,intermediary,T2,16,0.014814
71,intermediary,T2,17,0.014219


In [26]:

# 6) Construcción de candidatos: estrategia oficial por slice_number + fallback de revisión
def read_dicom(path):
    try:
        ds = pydicom.dcmread(str(path), force=True)
        _ = ds.pixel_array
        return ds
    except Exception:
        return None

img = image_df.copy()
gt = gt_png.copy()
img["instance_number_num"] = pd.to_numeric(img["instance_number"], errors="coerce").astype("Int64")
gt["disc_id_num"] = pd.to_numeric(gt["disc_or_slice_id"], errors="coerce").astype("Int64")

candidate_rows = []
if len(slices_long_df):
    sx = slices_long_df.copy()
    sx["case_id"] = sx["case_id"].apply(norm_case)
    sx["slice_number_num"] = pd.to_numeric(sx["slice_number"], errors="coerce").astype("Int64")
    sx["disc_id_num"] = pd.to_numeric(sx["disc_id"], errors="coerce").astype("Int64")
    sx["modality"] = sx["modality"].fillna("T1")

    m1 = img.merge(
        sx,
        left_on=["case_id_norm", "modality", "instance_number_num"],
        right_on=["case_id", "modality", "slice_number_num"],
        how="inner",
        suffixes=("_img", "_slice")
    )
    m2 = m1.merge(
        gt,
        left_on=["case_id_norm", "modality", "disc_id_num"],
        right_on=["case_id_norm", "modality", "disc_id_num"],
        how="inner",
        suffixes=("", "_gt")
    )
    for _, r in m2.iterrows():
        candidate_rows.append({
            "strategy": "official_slice_number",
            "case_id": r["case_id_norm"],
            "modality": r["modality"],
            "disc_id": int(r["disc_id_num"]) if pd.notna(r["disc_id_num"]) else None,
            "instance_number": int(r["instance_number_num"]) if pd.notna(r["instance_number_num"]) else None,
            "image_file_path": r["image_file_path"],
            "gt_file_path": r["gt_file_path"],
            "gt_type": r.get("gt_type"),
            "evidence": f"{r.get('source_table')} row={r.get('source_row')} col={r.get('source_column')} val={r.get('source_value')}",
        })

fallback = (
    img[img["case_id_norm"].notna() & img["modality"].notna()]
    .sort_values(["case_id_norm", "modality", "instance_number_num"])
    .groupby(["case_id_norm", "modality"], group_keys=False)
    .head(12)
    .merge(
        gt[gt["case_id_norm"].notna() & gt["modality"].notna() & gt["disc_id_num"].notna()]
        .sort_values(["case_id_norm", "modality", "gt_type", "disc_id_num"])
        .groupby(["case_id_norm", "modality", "disc_id_num"], group_keys=False)
        .head(2),
        on=["case_id_norm", "modality"],
        how="inner",
        suffixes=("_img", "_gt")
    )
)
for _, r in fallback.head(1500).iterrows():
    candidate_rows.append({
        "strategy": "fallback_case_modality_disc_review",
        "case_id": r["case_id_norm"],
        "modality": r["modality"],
        "disc_id": int(r["disc_id_num"]) if pd.notna(r["disc_id_num"]) else None,
        "instance_number": int(r["instance_number_num"]) if pd.notna(r["instance_number_num"]) else None,
        "image_file_path": r["image_file_path"],
        "gt_file_path": r["gt_file_path"],
        "gt_type": r.get("gt_type"),
        "evidence": "fallback; requiere revisión visual/manual",
    })

official_candidates_df = pd.DataFrame(candidate_rows).drop_duplicates(["image_file_path", "gt_file_path", "strategy"])
official_candidates_df.to_csv(E8_ROOT / "E8_official_pairing_candidates_raw.csv", index=False)
print("official_candidates_df:", official_candidates_df.shape)
display(official_candidates_df.head(50))
if len(official_candidates_df):
    display(official_candidates_df.groupby(["strategy", "modality", "gt_type"], dropna=False).size().reset_index(name="n").sort_values("n", ascending=False))


official_candidates_df: (1500, 9)


,strategy,case_id,modality,disc_id,instance_number,image_file_path,gt_file_path,gt_type,evidence
0,fallback_case_modality_disc_review,0001,T1,3,1,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0001/L-SPINE_LSS_20160309_091629_240000/T1_TSE_TRA_00...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T1_Output/T1_0001_D3.png,intermediary,fallback; requiere revisión visual/manual
1,fallback_case_modality_disc_review,0001,T1,4,1,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0001/L-SPINE_LSS_20160309_091629_240000/T1_TSE_TRA_00...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T1_Output/T1_0001_D4.png,intermediary,fallback; requiere revisión visual/manual
2,fallback_case_modality_disc_review,0001,T1,5,1,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0001/L-SPINE_LSS_20160309_091629_240000/T1_TSE_TRA_00...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T1_Output/T1_0001_D5.png,intermediary,fallback; requiere revisión visual/manual
3,fallback_case_modality_disc_review,0001,T1,3,1,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0001/L-SPINE_LSS_20160309_091629_240000/T1_TSE_TRA_00...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Manual_Label_Data/03_Manual_Label_Data/Labeller 01/T1_0001_D3.png,manual,fallback; requiere revisión visual/manual
4,fallback_case_modality_disc_review,0001,T1,4,1,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0001/L-SPINE_LSS_20160309_091629_240000/T1_TSE_TRA_00...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Manual_Label_Data/03_Manual_Label_Data/Labeller 01/T1_0001_D4.png,manual,fallback; requiere revisión visual/manual
5,fallback_case_modality_disc_review,0001,T1,5,1,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0001/L-SPINE_LSS_20160309_091629_240000/T1_TSE_TRA_00...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Manual_Label_Data/03_Manual_Label_Data/Labeller 01/T1_0001_D5.png,manual,fallback; requiere revisión visual/manual
6,fallback_case_modality_disc_review,0001,T1,3,2,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0001/L-SPINE_LSS_20160309_091629_240000/T1_TSE_TRA_00...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T1_Output/T1_0001_D3.png,intermediary,fallback; requiere revisión visual/manual
7,fallback_case_modality_disc_review,0001,T1,4,2,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0001/L-SPINE_LSS_20160309_091629_240000/T1_TSE_TRA_00...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T1_Output/T1_0001_D4.png,intermediary,fallback; requiere revisión visual/manual
8,fallback_case_modality_disc_review,0001,T1,5,2,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0001/L-SPINE_LSS_20160309_091629_240000/T1_TSE_TRA_00...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T1_Output/T1_0001_D5.png,intermediary,fallback; requiere revisión visual/manual
9,fallback_case_modality_disc_review,0001,T1,3,2,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/0

,strategy,modality,gt_type,n
0,fallback_case_modality_disc_review,T1,intermediary,504
1,fallback_case_modality_disc_review,T1,manual,504
2,fallback_case_modality_disc_review,T2,intermediary,492


In [27]:

# 7) Visualizar overlays por valor/clase, no como máscara binaria global
def normalize_image(arr):
    arr = arr.astype(np.float32)
    p1, p99 = np.percentile(arr, [1, 99])
    if p99 <= p1:
        return np.zeros_like(arr)
    return (np.clip(arr, p1, p99) - p1) / (p99 - p1 + 1e-8)

def mask_for_value(arr, value):
    if arr.ndim == 3:
        v = np.array(eval(value) if value.startswith("(") else value)
        return np.all(arr[..., :len(v)] == v, axis=-1).astype(np.uint8)
    return (arr == int(value)).astype(np.uint8)

def top_candidate_values(mask_path, min_ratio=0.0005, max_ratio=0.35, max_values=5):
    arr = read_mask_raw(mask_path)
    shape, prof = value_profile(mask_path, max_values=50)
    out = []
    # excluir valor dominante como background probable
    for item in prof[1:]:
        if min_ratio <= item["ratio"] <= max_ratio:
            out.append(item["value"])
        if len(out) >= max_values:
            break
    return arr, out

vis_source = official_candidates_df.copy()
vis_source["strategy_priority"] = vis_source["strategy"].map({"official_slice_number": 0, "fallback_case_modality_disc_review": 1}).fillna(9)
vis_source["modality_priority"] = vis_source["modality"].map({"T1": 0, "T2": 1}).fillna(9)
vis_source["gt_priority"] = vis_source["gt_type"].map({"final": 0, "intermediary": 1, "manual": 2}).fillna(9)
vis_df = (
    vis_source.sort_values(["strategy_priority", "modality_priority", "gt_priority", "case_id", "disc_id", "instance_number"])
    .groupby(["case_id", "modality", "disc_id"], group_keys=False)
    .head(2)
    .head(120)
    .copy()
)

review_rows = []
print("Candidatos a visualizar:", len(vis_df))
for i, (_, r) in enumerate(tqdm(vis_df.iterrows(), total=len(vis_df), desc="official overlays"), start=1):
    cid = f"official_candidate_{i:03d}"
    try:
        ds = read_dicom(r["image_file_path"])
        if ds is None:
            raise RuntimeError("No se pudo leer DICOM")
        img_pixels = ds.pixel_array.astype(np.float32)
        img_norm = normalize_image(img_pixels)

        raw_mask, values = top_candidate_values(r["gt_file_path"])
        ncols = 3 + min(len(values), 4)
        fig, axes = plt.subplots(1, ncols, figsize=(4*ncols, 4))
        axes[0].imshow(img_norm, cmap="gray"); axes[0].set_title("Axial IMA"); axes[0].axis("off")
        axes[1].imshow(raw_mask if raw_mask.ndim == 3 else raw_mask, cmap=None if raw_mask.ndim == 3 else "nipy_spectral")
        axes[1].set_title("Raw GT"); axes[1].axis("off")

        combined = np.zeros(raw_mask.shape[:2], dtype=np.uint8)
        value_stats = []
        for v in values:
            m = mask_for_value(raw_mask, v)
            if m.shape != img_pixels.shape:
                m = resize(m.astype(np.float32), img_pixels.shape, order=0, preserve_range=True, anti_aliasing=False) > 0.5
                m = m.astype(np.uint8)
            ratio = float(m.mean())
            _, comps = ndimage.label(m > 0)
            combined = np.maximum(combined, m)
            value_stats.append({"value": v, "ratio": ratio, "components": int(comps)})

        axes[2].imshow(img_norm, cmap="gray")
        axes[2].imshow(np.ma.masked_where(combined <= 0, combined), alpha=0.45, cmap="autumn")
        axes[2].set_title("Overlay valores candidatos"); axes[2].axis("off")

        for ax, stat in zip(axes[3:], value_stats[:4]):
            m = mask_for_value(raw_mask, stat["value"])
            if m.shape != img_pixels.shape:
                m = resize(m.astype(np.float32), img_pixels.shape, order=0, preserve_range=True, anti_aliasing=False) > 0.5
            ax.imshow(img_norm, cmap="gray")
            ax.imshow(np.ma.masked_where(m <= 0, m), alpha=0.55, cmap="autumn")
            ax.set_title(f"value={stat['value']}\nratio={stat['ratio']:.3f}")
            ax.axis("off")

        fig.suptitle(
            f"{cid} | {r['strategy']} | case={r['case_id']} | {r['modality']} | D={r['disc_id']} | inst={r['instance_number']} | gt={r['gt_type']}",
            fontsize=10
        )
        fig.tight_layout()
        fig_path = FIGURES_ROOT / f"E8_alkafri_official_pairing_{i:03d}.png"
        fig.savefig(fig_path, dpi=150, bbox_inches="tight")
        plt.close(fig)

        review_rows.append({
            "candidate_id": cid,
            "figure_path": str(fig_path),
            "strategy": r["strategy"],
            "case_id": r["case_id"],
            "modality": r["modality"],
            "disc_id": r["disc_id"],
            "instance_number": r["instance_number"],
            "gt_type": r["gt_type"],
            "image_file_path": r["image_file_path"],
            "gt_file_path": r["gt_file_path"],
            "candidate_values": json.dumps(value_stats, ensure_ascii=False),
            "auto_status": "review",
            "manual_accept": "",
            "manual_reject_reason": "",
            "notes": "",
        })
    except Exception as exc:
        review_rows.append({
            "candidate_id": cid,
            "strategy": r.get("strategy"),
            "case_id": r.get("case_id"),
            "modality": r.get("modality"),
            "disc_id": r.get("disc_id"),
            "instance_number": r.get("instance_number"),
            "gt_type": r.get("gt_type"),
            "image_file_path": r.get("image_file_path"),
            "gt_file_path": r.get("gt_file_path"),
            "auto_status": "error",
            "error": repr(exc),
            "manual_accept": "",
            "manual_reject_reason": "",
            "notes": "",
        })

review_df = pd.DataFrame(review_rows)
review_df.to_csv(E8_ROOT / "E8_official_pairing_visual_review_sheet.csv", index=False)
print("Figuras generadas:", int(review_df["figure_path"].notna().sum()) if "figure_path" in review_df else 0)
display(review_df.head(30))


Candidatos a visualizar: 120


official overlays:   0%|          | 0/120 [00:00<?, ?it/s]

Figuras generadas: 120


,candidate_id,figure_path,strategy,case_id,modality,disc_id,instance_number,gt_type,image_file_path,gt_file_path,candidate_values,auto_status,manual_accept,manual_reject_reason,notes
0,official_candidate_001,/content/drive/MyDrive/PFI_MVP/figures/E8_alkafri_official_pairing_001.png,fallback_case_modality_disc_review,0001,T1,3,1,intermediary,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0001/L-SPINE_LSS_20160309_091629_240000/T1_TSE_TRA_00...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T1_Output/T1_0001_D3.png,"[{""value"": ""8"", ""ratio"": 0.037509765625, ""components"": 2677}, {""value"": ""10"", ""ratio"": 0.03505859375, ""components"": 2688}, {""value"": ""11"", ""ratio"": 0.032021...",review,,,
1,official_candidate_002,/content/drive/MyDrive/PFI_MVP/figures/E8_alkafri_official_pairing_002.png,fallback_case_modality_disc_review,0001,T1,3,2,intermediary,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0001/L-SPINE_LSS_20160309_091629_240000/T1_TSE_TRA_00...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T1_Output/T1_0001_D3.png,"[{""value"": ""8"", ""ratio"": 0.037509765625, ""components"": 2677}, {""value"": ""10"", ""ratio"": 0.03505859375, ""components"": 2688}, {""value"": ""11"", ""ratio"": 0.032021...",review,,,
2,official_candidate_003,/content/drive/MyDrive/PFI_MVP/figures/E8_alkafri_official_pairing_003.png,fallback_case_modality_disc_review,0001,T1,4,1,intermediary,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0001/L-SPINE_LSS_20160309_091629_240000/T1_TSE_TRA_00...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T1_Output/T1_0001_D4.png,"[{""value"": ""5"", ""ratio"": 0.04857421875, ""components"": 2681}, {""value"": ""6"", ""ratio"": 0.0439453125, ""components"": 2966}, {""value"": ""7"", ""ratio"": 0.0428027343...",review,,,
3,official_candidate_004,/content/drive/MyDrive/PFI_MVP/figures/E8_alkafri_official_pairing_004.png,fallback_case_modality_disc_review,0001,T1,4,2,intermediary,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0001/L-SPINE_LSS_20160309_091629_240000/T1_TSE_TRA_00...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T1_Output/T1_0001_D4.png,"[{""value"": ""5"", ""ratio"": 0.04857421875, ""components"": 2681}, {""value"": ""6"", ""ratio"": 0.0439453125, ""components"": 2966}, {""value"": ""7"", ""ratio"": 0.0428027343...",review,,,
4,official_candidate_005,/content/drive/MyDrive/PFI_MVP/figures/E8_alkafri_official_pairing_005.png,fallback_case_modality_disc_review,0001,T1,5,1,intermediary,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0001/L-SPINE_LSS_20160309_091629_240000/T1_TSE_TRA_00...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T1_Output/T1_0001_D5.png,"[{""value"": ""5"", ""ratio"": 0.03935546875, ""components"": 2251}, {""value"": ""6"", ""ratio"": 0.036318359375, ""components"": 2282}, {""value"": ""7"", ""ratio"": 0.03086914...",review,,,
5,official_candidate_006,/content/drive/MyDrive/PFI_MVP/figures/E8_alkafri_official_pairing_006.png,fallback_case_modality_disc_review,0001,T1,5,2,intermediary,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0001/L-SPINE_LSS_20160309_091629_240000/T1_TSE_TRA_00...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Tru

In [28]:

# 8) Reporte técnico y decisión
n_official = int((official_candidates_df["strategy"] == "official_slice_number").sum()) if len(official_candidates_df) else 0
n_fallback = int((official_candidates_df["strategy"] == "fallback_case_modality_disc_review").sum()) if len(official_candidates_df) else 0
n_figs = int(review_df["figure_path"].notna().sum()) if "figure_path" in review_df else 0

decision = "pending_manual_review"
recommendation = (
    "Abrir E8_official_pairing_visual_review_sheet.csv y marcar manual_accept=yes "
    "solo si el overlay por valor/clase cae anatómicamente sobre estructuras esperadas. "
    "Si hay >=30 aceptados, crear subset axial curado. Si no, cerrar Al-Kafri como no validado."
)

report = {
    "notebook": "16_E8_alkafri_official_pairing_and_mask_decode",
    "goal": "rescue axial segmentation pairing using official metadata and mask value profiling",
    "metadata_tables_found": list(metadata_tables.keys()),
    "slices_long_rows": int(len(slices_long_df)),
    "image_index_rows": int(len(image_df)),
    "gt_index_rows": int(len(gt_df)),
    "gt_png_rows": int(len(gt_png)),
    "official_slice_number_candidates": n_official,
    "fallback_review_candidates": n_fallback,
    "visual_review_figures": n_figs,
    "decision": decision,
    "recommendation": recommendation,
}

(E8_ROOT / "E8_alkafri_official_pairing_report.json").write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")

lines = [
    "# E8 Al-Kafri official pairing and mask decode rescue",
    "",
    "## Objetivo",
    "Intentar rescatar el dataset axial Al-Kafri/Sudirman usando metadata oficial y perfiles reales de valores de máscara.",
    "",
    "## Resultados",
    f"- Metadata CSV encontrada: {len(metadata_tables)}",
    f"- Filas detectadas en Slices Numbers long: {len(slices_long_df)}",
    f"- Imágenes indexadas: {len(image_df)}",
    f"- GT indexados: {len(gt_df)}",
    f"- GT PNG: {len(gt_png)}",
    f"- Candidatos por metadata oficial slice number: {n_official}",
    f"- Candidatos fallback para revisión: {n_fallback}",
    f"- Figuras para revisión visual: {n_figs}",
    "",
    "## Decisión",
    decision,
    "",
    "## Recomendación",
    recommendation,
]
(DOCS_ROOT / "E8_alkafri_official_pairing_conclusion.md").write_text("\n".join(lines), encoding="utf-8")
print(json.dumps(report, indent=2, ensure_ascii=False))


{
  "notebook": "16_E8_alkafri_official_pairing_and_mask_decode",
  "goal": "rescue axial segmentation pairing using official metadata and mask value profiling",
  "metadata_tables_found": [
    "Slices Numbers.csv",
    "T1_Subfolders.csv",
    "T2_Subfolders.csv",
    "PNG counts.csv",
    "misregistration.csv"
  ],
  "slices_long_rows": 1595,
  "image_index_rows": 8150,
  "gt_index_rows": 37080,
  "gt_png_rows": 18540,
  "official_slice_number_candidates": 0,
  "fallback_review_candidates": 1500,
  "visual_review_figures": 120,
  "decision": "pending_manual_review",
  "recommendation": "Abrir E8_official_pairing_visual_review_sheet.csv y marcar manual_accept=yes solo si el overlay por valor/clase cae anatómicamente sobre estructuras esperadas. Si hay >=30 aceptados, crear subset axial curado. Si no, cerrar Al-Kafri como no validado."
}


In [29]:
# E8 SAFE VISUALIZATION - usar SOLO candidatos oficiales filtrados
# Esta celda NO depende de official_candidates_df en memoria.
# Carga directamente el CSV bueno y genera figuras nuevas con nombre OFFICIAL.

import ast
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pydicom
from PIL import Image
from scipy import ndimage
from skimage.transform import resize
from tqdm.auto import tqdm

OFFICIAL_FILTERED_PATH = E8_ROOT / "E8_official_pairing_candidates_HEADER_NONE_FILTERED.csv"

official_candidates_df = pd.read_csv(OFFICIAL_FILTERED_PATH)

print("Candidatos cargados desde CSV oficial filtrado:", official_candidates_df.shape)
display(
    official_candidates_df
    .groupby(["strategy", "modality", "gt_type", "slice_offset"], dropna=False)
    .size()
    .reset_index(name="n")
)

assert len(official_candidates_df) > 0
assert official_candidates_df["strategy"].astype(str).str.contains("official_slice_number_header_none").all()

def read_dicom(path):
    ds = pydicom.dcmread(str(path), force=True)
    _ = ds.pixel_array
    return ds

def normalize_image(arr):
    arr = arr.astype(np.float32)
    p1, p99 = np.percentile(arr, [1, 99])
    if p99 <= p1:
        return np.zeros_like(arr)
    return (np.clip(arr, p1, p99) - p1) / (p99 - p1 + 1e-8)

def read_mask_raw(path):
    with Image.open(path) as img:
        arr = np.asarray(img)
    if arr.ndim == 3 and arr.shape[-1] == 4:
        arr = arr[..., :3]
    return arr

def value_profile(mask_path, max_values=40):
    arr = read_mask_raw(mask_path)

    if arr.ndim == 3:
        flat = arr.reshape(-1, arr.shape[-1])
        vals, counts = np.unique(flat, axis=0, return_counts=True)
        values = [tuple(map(int, v.tolist())) for v in vals]
    else:
        vals, counts = np.unique(arr.reshape(-1), return_counts=True)
        values = [int(v) for v in vals]

    total = int(np.prod(arr.shape[:2]))
    order = np.argsort(counts)[::-1]

    prof = []
    for idx in order[:max_values]:
        prof.append({
            "value": values[idx],
            "count": int(counts[idx]),
            "ratio": float(counts[idx] / total),
        })

    return arr.shape, prof

def parse_value(v):
    if isinstance(v, (int, np.integer)):
        return int(v)

    if isinstance(v, tuple):
        return v

    s = str(v)

    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, tuple):
            return tuple(int(x) for x in parsed)
        return int(parsed)
    except Exception:
        return int(float(s))

def mask_for_value(arr, value):
    v = parse_value(value)

    if arr.ndim == 3:
        v_arr = np.array(v)
        return np.all(arr[..., :len(v_arr)] == v_arr, axis=-1).astype(np.uint8)

    return (arr == int(v)).astype(np.uint8)

def top_candidate_values(mask_path, min_ratio=0.0005, max_ratio=0.20, max_values=5):
    arr = read_mask_raw(mask_path)
    _, prof = value_profile(mask_path, max_values=60)

    # Excluimos el valor más dominante como probable fondo.
    values = []
    for item in prof[1:]:
        if min_ratio <= item["ratio"] <= max_ratio:
            values.append(item["value"])
        if len(values) >= max_values:
            break

    return arr, values

# Selección balanceada para revisar: por modalidad y disco.
vis_df = (
    official_candidates_df
    .sort_values(["case_id", "modality", "disc_id", "instance_number"])
    .groupby(["modality", "disc_id"], group_keys=False)
    .head(25)
    .head(120)
    .copy()
)

print("Candidatos oficiales a visualizar:", len(vis_df))

review_rows = []

for i, (_, r) in enumerate(tqdm(vis_df.iterrows(), total=len(vis_df), desc="official HEADER_NONE overlays"), start=1):
    cid = f"official_header_none_{i:03d}"

    try:
        ds = read_dicom(r["image_file_path"])
        img_pixels = ds.pixel_array.astype(np.float32)
        img_norm = normalize_image(img_pixels)

        raw_mask, values = top_candidate_values(r["gt_file_path"])

        ncols = 3 + min(len(values), 4)
        fig, axes = plt.subplots(1, ncols, figsize=(4 * ncols, 4))

        axes[0].imshow(img_norm, cmap="gray")
        axes[0].set_title("Axial IMA")
        axes[0].axis("off")

        axes[1].imshow(raw_mask if raw_mask.ndim == 3 else raw_mask, cmap=None if raw_mask.ndim == 3 else "nipy_spectral")
        axes[1].set_title("Raw GT")
        axes[1].axis("off")

        combined = np.zeros(raw_mask.shape[:2], dtype=np.uint8)
        value_stats = []

        for v in values:
            m = mask_for_value(raw_mask, v)

            if m.shape != img_pixels.shape:
                m = resize(
                    m.astype(np.float32),
                    img_pixels.shape,
                    order=0,
                    preserve_range=True,
                    anti_aliasing=False,
                ) > 0.5
                m = m.astype(np.uint8)

            ratio = float(m.mean())
            _, comps = ndimage.label(m > 0)

            combined = np.maximum(combined, m)

            value_stats.append({
                "value": str(v),
                "ratio": ratio,
                "components": int(comps),
            })

        axes[2].imshow(img_norm, cmap="gray")
        axes[2].imshow(np.ma.masked_where(combined <= 0, combined), alpha=0.45, cmap="autumn")
        axes[2].set_title("Overlay valores candidatos")
        axes[2].axis("off")

        for ax, stat in zip(axes[3:], value_stats[:4]):
            m = mask_for_value(raw_mask, stat["value"])

            if m.shape != img_pixels.shape:
                m = resize(
                    m.astype(np.float32),
                    img_pixels.shape,
                    order=0,
                    preserve_range=True,
                    anti_aliasing=False,
                ) > 0.5

            ax.imshow(img_norm, cmap="gray")
            ax.imshow(np.ma.masked_where(m <= 0, m), alpha=0.55, cmap="autumn")
            ax.set_title(f"value={stat['value']}\nratio={stat['ratio']:.3f}")
            ax.axis("off")

        fig.suptitle(
            f"{cid} | {r['strategy']} | case={r['case_id']} | {r['modality']} | "
            f"D={r['disc_id']} | slice={r['slice_number']} | inst={r['instance_number']} | gt={r['gt_type']}",
            fontsize=10,
        )

        fig.tight_layout()

        fig_path = FIGURES_ROOT / f"E8_alkafri_OFFICIAL_HEADER_NONE_{i:03d}.png"
        fig.savefig(fig_path, dpi=150, bbox_inches="tight")
        plt.close(fig)

        review_rows.append({
            "candidate_id": cid,
            "figure_path": str(fig_path),
            "strategy": r["strategy"],
            "case_id": r["case_id"],
            "modality": r["modality"],
            "disc_id": r["disc_id"],
            "slice_number": r["slice_number"],
            "slice_offset": r["slice_offset"],
            "instance_number": r["instance_number"],
            "gt_type": r["gt_type"],
            "image_file_path": r["image_file_path"],
            "gt_file_path": r["gt_file_path"],
            "candidate_values": json.dumps(value_stats, ensure_ascii=False),
            "auto_status": "review",
            "manual_accept": "",
            "manual_reject_reason": "",
            "notes": "",
        })

    except Exception as exc:
        review_rows.append({
            "candidate_id": cid,
            "strategy": r.get("strategy"),
            "case_id": r.get("case_id"),
            "modality": r.get("modality"),
            "disc_id": r.get("disc_id"),
            "slice_number": r.get("slice_number"),
            "slice_offset": r.get("slice_offset"),
            "instance_number": r.get("instance_number"),
            "gt_type": r.get("gt_type"),
            "image_file_path": r.get("image_file_path"),
            "gt_file_path": r.get("gt_file_path"),
            "auto_status": "error",
            "error": repr(exc),
            "manual_accept": "",
            "manual_reject_reason": "",
            "notes": "",
        })

review_df = pd.DataFrame(review_rows)

review_path = E8_ROOT / "E8_official_HEADER_NONE_visual_review_sheet.csv"
review_df.to_csv(review_path, index=False)

print("Figuras oficiales generadas:", review_df["figure_path"].notna().sum() if "figure_path" in review_df.columns else 0)
print("Review sheet:", review_path)

display(review_df.head(30))

assert review_df["strategy"].astype(str).str.contains("official_slice_number_header_none").all()
print("OK: las figuras generadas corresponden a candidatos OFICIALES, no fallback.")

Candidatos cargados desde CSV oficial filtrado: (1201, 13)


,strategy,modality,gt_type,slice_offset,n
0,official_slice_number_header_none,T1,intermediary,0,591
1,official_slice_number_header_none,T2,intermediary,0,610


Candidatos oficiales a visualizar: 120


official HEADER_NONE overlays:   0%|          | 0/120 [00:00<?, ?it/s]

Figuras oficiales generadas: 120
Review sheet: /content/drive/MyDrive/PFI_MVP/results/E8_alkafri_official_pairing/E8_official_HEADER_NONE_visual_review_sheet.csv


,candidate_id,figure_path,strategy,case_id,modality,disc_id,slice_number,slice_offset,instance_number,gt_type,image_file_path,gt_file_path,candidate_values,auto_status,manual_accept,manual_reject_reason,notes
0,official_header_none_001,/content/drive/MyDrive/PFI_MVP/figures/E8_alkafri_OFFICIAL_HEADER_NONE_001.png,official_slice_number_header_none,1,T1,3,3,0,3,intermediary,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0001/L-SPINE_LSS_20160309_091629_240000/T1_TSE_TRA_00...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T1_Output/T1_0001_D3.png,"[{""value"": ""8"", ""ratio"": 0.037509765625, ""components"": 2677}, {""value"": ""10"", ""ratio"": 0.03505859375, ""components"": 2688}, {""value"": ""11"", ""ratio"": 0.032021...",review,,,
1,official_header_none_002,/content/drive/MyDrive/PFI_MVP/figures/E8_alkafri_OFFICIAL_HEADER_NONE_002.png,official_slice_number_header_none,1,T1,4,7,0,7,intermediary,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0001/L-SPINE_LSS_20160309_091629_240000/T1_TSE_TRA_00...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T1_Output/T1_0001_D4.png,"[{""value"": ""5"", ""ratio"": 0.04857421875, ""components"": 2681}, {""value"": ""6"", ""ratio"": 0.0439453125, ""components"": 2966}, {""value"": ""7"", ""ratio"": 0.0428027343...",review,,,
2,official_header_none_003,/content/drive/MyDrive/PFI_MVP/figures/E8_alkafri_OFFICIAL_HEADER_NONE_003.png,official_slice_number_header_none,1,T1,5,11,0,11,intermediary,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0001/L-SPINE_LSS_20160309_091629_240000/T1_TSE_TRA_00...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T1_Output/T1_0001_D5.png,"[{""value"": ""5"", ""ratio"": 0.03935546875, ""components"": 2251}, {""value"": ""6"", ""ratio"": 0.036318359375, ""components"": 2282}, {""value"": ""7"", ""ratio"": 0.03086914...",review,,,
3,official_header_none_004,/content/drive/MyDrive/PFI_MVP/figures/E8_alkafri_OFFICIAL_HEADER_NONE_004.png,official_slice_number_header_none,1,T2,3,3,0,3,intermediary,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0001/L-SPINE_LSS_20160309_091629_240000/T2_TSE_TRA_38...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T2_Output/T2_0001_D3.png,"[{""value"": ""10"", ""ratio"": 0.052919921875, ""components"": 3916}, {""value"": ""12"", ""ratio"": 0.044912109375, ""components"": 3541}, {""value"": ""9"", ""ratio"": 0.03676...",review,,,
4,official_header_none_005,/content/drive/MyDrive/PFI_MVP/figures/E8_alkafri_OFFICIAL_HEADER_NONE_005.png,official_slice_number_header_none,1,T2,4,7,0,7,intermediary,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0001/L-SPINE_LSS_20160309_091629_240000/T2_TSE_TRA_38...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T2_Output/T2_0001_D4.png,"[{""value"": ""7"", ""ratio"": 0.057646484375, ""components"": 4209}, {""value"": ""9"", ""ratio"": 0.055283203125, ""components"": 4273}, {""value"": ""10"", ""ratio"": 0.048691...",review,,,
5,official_header_none_006,/content/drive/MyDrive/PFI_MVP/figures/E8_alkafri_OFFICIAL_HEADER_NONE_006.png,official_slice_number_header_none,1,T2,5,11,0,11,intermediary,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0001/L-SPINE_LSS_20160309_091629_240000/T2_TSE_TRA_38...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_

OK: las figuras generadas corresponden a candidatos OFICIALES, no fallback.


In [30]:
# E8 SAFE REPORT - reporte corregido usando candidatos oficiales filtrados

official_filtered_path = E8_ROOT / "E8_official_pairing_candidates_HEADER_NONE_FILTERED.csv"
review_path = E8_ROOT / "E8_official_HEADER_NONE_visual_review_sheet.csv"

official_filtered_df = pd.read_csv(official_filtered_path)
review_df = pd.read_csv(review_path)

report = {
    "notebook": "16_E8_alkafri_official_pairing_and_mask_decode",
    "goal": "rescue axial segmentation pairing using official metadata and mask value profiling",
    "decision": "pending_visual_review",
    "official_pairing_source": "Slices Numbers.csv read with header=None + T1_Subfolders.csv + T2_Subfolders.csv",
    "official_candidates_filtered": int(len(official_filtered_df)),
    "official_candidates_by_modality_gt": (
        official_filtered_df
        .groupby(["modality", "gt_type", "slice_offset"], dropna=False)
        .size()
        .reset_index(name="n")
        .to_dict(orient="records")
    ),
    "visual_review_figures": int(review_df["figure_path"].notna().sum()) if "figure_path" in review_df.columns else int(len(review_df)),
    "review_sheet": str(review_path),
    "candidate_csv": str(official_filtered_path),
    "recommendation": (
        "Revisar visualmente las figuras E8_alkafri_OFFICIAL_HEADER_NONE_*.png. "
        "Si al menos 30 overlays muestran correspondencia anatómica razonable, crear subset axial curado. "
        "Si no, documentar que el pairing oficial fue reconstruido pero la decodificación/overlay no fue suficiente para entrenamiento."
    ),
}

report_path = E8_ROOT / "E8_alkafri_official_HEADER_NONE_report.json"
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")

print(json.dumps(report, indent=2, ensure_ascii=False))
print("Reporte corregido:", report_path)

{
  "notebook": "16_E8_alkafri_official_pairing_and_mask_decode",
  "goal": "rescue axial segmentation pairing using official metadata and mask value profiling",
  "decision": "pending_visual_review",
  "official_pairing_source": "Slices Numbers.csv read with header=None + T1_Subfolders.csv + T2_Subfolders.csv",
  "official_candidates_filtered": 1201,
  "official_candidates_by_modality_gt": [
    {
      "modality": "T1",
      "gt_type": "intermediary",
      "slice_offset": 0,
      "n": 591
    },
    {
      "modality": "T2",
      "gt_type": "intermediary",
      "slice_offset": 0,
      "n": 610
    }
  ],
  "visual_review_figures": 120,
  "review_sheet": "/content/drive/MyDrive/PFI_MVP/results/E8_alkafri_official_pairing/E8_official_HEADER_NONE_visual_review_sheet.csv",
  "candidate_csv": "/content/drive/MyDrive/PFI_MVP/results/E8_alkafri_official_pairing/E8_official_pairing_candidates_HEADER_NONE_FILTERED.csv",
  "recommendation": "Revisar visualmente las figuras E8_alkafri_OFF